# Compute Forward Operator

## 📦 Imports

In [3]:
import mindmap as mm 
import mne
#import pyvista as pv

# readers
dm = mm.io.TutoManager("../../config.yml")
sr = dm.get_subject("intra-12")

# data
ave   = sr.load("ave", task="spk")
trans = sr.load("eeg-trans")
src   = sr.load("src")
bem   = sr.load("bem")

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- `bem`: Boundary Element Model (Head conductor model).
- `src-raw`: Source space (before computing forward).
- `eeg`: EEG.
- `trans`: Affine transformation for EEG coregistration.

🎚️ **<font color=pink>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)
- `hmodel`: Type of head model ["bem", "bem-sim"]. Whether to compute the forward solution for inverse problems or simulations.

In [ ]:
# # ------------- Parameters ---------------
# fconfig = "../../data/config.yml"
# sub     = "intra-12"
# hmodel  = "bem"
# task    = "spk"
# # ----------------------------------------

# reader = mm.io.TutoReader(fconfig)
# mm_dir = reader.config["MM_DIR"]

# # data
# data = reader(sub, hmodel, "src-raw", "ave", "trans", task=task)

AttributeError: module 'mindmap.io' has no attribute 'TutoReader'

📊 **Check coregistration**

In [4]:
mm.viz.plot_alignment(bem, ave).show()

Widget(value='<iframe src="http://localhost:36717/index.html?ui=P_0x718f11bad9d0_0&reconnect=auto" class="pyvi…

## ⚙️ Compute Forward Solution

In [5]:
forward = mne.make_forward_solution(
    info    = ave.info,
    trans   = trans,
    src     = src,
    bem     = bem,
    meg     = False,
    eeg     = True,
    verbose = "warning",
)

📊 **Plot leadfields**

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_leadfield_summary(forward : mne.Forward):
    """
    Plot a summary of the leadfield matrix properties.
    """
    gain = forward["sol"]["data"]
    n_channels, n_sources = gain.shape

    # 1. Calculate statistics
    col_sums  = np.sum(gain, axis=0)
    row_means = np.mean(gain, axis=1)

    # check for average reference (sums shoulb be near zero)
    is_avg_ref = np.allclose(col_sums, 0, atol=1e-12)

    # setup plotting grid
    fig, axes = plt.subplots(2, 2, figsize=(14,10))
    fig.suptitle(f"Leadfield Summary\nChannels: {n_channels} | Sources: {n_sources}", fontsize=16)

    # -----------------------------------------
    # Plot A: Distribution of Leadfield Values
    # -----------------------------------------

    sns.histplot(gain.ravel(), bins=100, ax=axes[0,0], kde=True, color="teal")
    axes[0, 0].set_title("Distribution of Leadfield Weights")
    axes[0, 0].set_xlabel("Source Index")
    axes[0, 0].set_ylabel("Frequency")

    # --------------------------------------
    # Plot B: Column Sums (Reference Check)
    # --------------------------------------

    axes[0, 1].plot(col_sums, color="coral", alpha=0.7)
    axes[0, 1].set_title(f"Sum of Columns (Avg Ref: {is_avg_ref})")
    axes[0, 1].set_xlabel("Source Index")
    axes[0, 1].set_ylabel("Sum")
    axes[0, 1].axhline(0, )

SyntaxError: incomplete input (2014216768.py, line 1)

In [ ]:
mm.viz.plot_leadfield_summary(forward)

In [6]:
mm.viz.plot_leadfield(ave, forward).show()

    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]


Widget(value='<iframe src="http://localhost:36717/index.html?ui=P_0x718efb26f190_1&reconnect=auto" class="pyvi…

💾 **Save Forward Solution** <br>
🎚️ **<font color=pink>Save</font>**:
- `overwrite`: Overwrite existing forward solution.


In [ ]:
# ------------ Parameters -------------
overwrite = True

file = mm.io.get_bidspath(
    root        = reader.config["MM_DIR"],
    subject     = sub,
    datatype    = "fwd",
    task        = None,
    session     = None,
    description = None,
)
# -------------------------------------

file.parent.mkdir(parents=True, exist_ok=True)

mne.write_forward_solution(file, forward, overwrite=overwrite, verbose="warning")
print(f"\033[92mSuccessfully saved forward solution for subject {data['sub']}\033[0m")

Successfully saved forward solution for subject intra-12
